In [ ]:
# Cell 1 — Install dependencies
!pip install pdfminer.six doclayout-yolo pymupdf openai huggingface_hub portkey-ai pillow -q

### Imports & config

In [1]:

import asyncio
import base64
import json
import re
from io import BytesIO
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import os

import fitz
from pdfminer.high_level import extract_pages
from pdfminer.layout import LTTextBox
from doclayout_yolo import YOLOv10
from huggingface_hub import hf_hub_download
from portkey_ai import Portkey
from PIL import Image

# DocLayout-YOLO class ids: 0=title,1=plain-text,2=abandon,3=figure,4=figure-caption,
# 5=table,6=table-caption,7=table-footnote,8=isolate-formula,9=formula-caption
VISUAL_CLASSES = {3: "Figure", 4: "Figure", 5: "Table", 6: "Table"}
CAPTION_CLASSES = {4, 6}

SYSTEM_PROMPT = (
    "You are a document analysis assistant. Your job is to classify and describe visual elements "
    "extracted from financial/technical documents with precision.\n"
    "You always respond in the exact structured format requested — no preamble, no extra text."
)

USER_PROMPT_TEMPLATE = """\
Caption extracted from document: '{caption}'

## Step 1 — Classify
Decide if this is a TABLE (rows/columns of data) or a FIGURE (chart, graph, diagram, image, plot).

## Step 2 — Extract identifier
From the caption text extract:
- The heading exactly as written (e.g. 'Exhibit 1', 'Figure 3'). This becomes the HEADING field.
- The number only (e.g. 3 from 'Figure 3'). If none found, leave blank.
- Sub-label only if explicitly named (e.g. '(a)'). If none, leave blank.

## Step 3 — Detect sub-plots
Count distinct plots/charts/panels visible.
- ONE plot → single figure, no sub-labels.
- MULTIPLE plots → label them (a), (b), (c)... left-to-right, top-to-bottom.

## Step 4 — Respond in EXACTLY this format, nothing else:

KEY: <TABLE or FIGURE>_<number>
HEADING: <exact caption heading>
CONTENT:
<your content — see rules below>

## Rules for CONTENT:
### If TABLE:
- Reproduce as a faithful markdown table. Preserve all rows, columns, headers and number formatting exactly.
- Do not summarize or drop any cells.

### If FIGURE with a single plot:
- Summarize strictly what is visible. Reference title, axes, legend, annotated numbers.
- Write 2-5 sentences in markdown.

### If FIGURE with multiple sub-plots:
- One overall sentence summarising the combined figure.
- Then each sub-plot: **(a)** <2-3 sentences> **(b)** <2-3 sentences> ...
- KEY stays as the parent e.g. FIGURE_4.
"""

C:\Users\78235\AppData\Roaming\Python\Python312\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\78235\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### PDF path & client setup

In [2]:
PDF_PATH = Path("..\\..\\Sample PDFs\\OP-RAG.pdf")

OUTPUT_DIR = "Output"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

client = Portkey(
    api_key=os.getenv("PORTKEY_API_KEY"),
    base_url=os.getenv("PORTKEY_BASE_URL"),
)

print(f"Using: {PDF_PATH}")

Using: ..\..\Sample PDFs\OP-RAG.pdf


### Load DocLayout-YOLO model

In [3]:
model_path = hf_hub_download(
    repo_id="juliozhao/DocLayout-YOLO-DocStructBench",
    filename="doclayout_yolo_docstructbench_imgsz1024.pt",
)
model = YOLOv10(model_path)
print("Model ready")

Model ready


### Image utilities

In [4]:
def page_to_pil(pdf_path: Path, page_index: int, dpi: int = 150) -> Image.Image:
    """Render a PDF page to a PIL Image."""
    doc = fitz.open(str(pdf_path))
    page = doc[page_index]
    pix = page.get_pixmap(matrix=fitz.Matrix(dpi / 72, dpi / 72))
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


def crop_pil(img: Image.Image, box_norm: tuple) -> Image.Image:
    """Crop PIL image using a normalised (0-1) bounding box."""
    W, H = img.size
    x1, y1, x2, y2 = box_norm
    return img.crop((int(x1 * W), int(y1 * H), int(x2 * W), int(y2 * H)))


def pil_to_b64(img: Image.Image, fmt: str = "PNG") -> str:
    """Encode a PIL Image to a base64 string."""
    buf = BytesIO()
    img.save(buf, format=fmt)
    return base64.b64encode(buf.getvalue()).decode()

### Detection helpers

In [5]:
def detect_visual_boxes(img: Image.Image, conf: float = 0.25) -> list[dict]:
    """Run DocLayout-YOLO; return list of detected visual element dicts."""
    W, H = img.size
    detections = []
    for r in model.predict(img, imgsz=1024, conf=conf, device="cpu"):
        for box in r.boxes:
            cls = int(box.cls)
            if cls not in VISUAL_CLASSES:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            detections.append({
                "class_id": cls,
                "label":    VISUAL_CLASSES[cls],
                "box_norm": (x1 / W, y1 / H, x2 / W, y2 / H),
            })
    return detections


def merge_figure_and_caption(detections: list[dict]) -> list[dict]:
    """Merge each figure/table detection with its nearest spatially-aligned caption."""
    figures  = [d for d in detections if d["class_id"] in {3, 5}]
    captions = [d for d in detections if d["class_id"] in {4, 6}]
    merged, used = [], set()

    for fig in figures:
        fx1, fy1, fx2, fy2 = fig["box_norm"]
        best_cap, best_dist = None, float("inf")
        for j, cap in enumerate(captions):
            cx1, cy1, cx2, cy2 = cap["box_norm"]
            h_overlap = min(fx2, cx2) - max(fx1, cx1)
            v_dist    = max(fy1 - cy2, cy1 - fy2, 0)
            if h_overlap > 0 and v_dist < 0.15 and v_dist < best_dist:
                best_dist, best_cap = v_dist, (j, cap)
        if best_cap:
            j, cap = best_cap
            used.add(j)
            cx1, cy1, cx2, cy2 = cap["box_norm"]
            merged_box = (min(fx1, cx1), min(fy1, cy1), max(fx2, cx2), max(fy2, cy2))
            merged.append({**fig, "box_norm": merged_box, "_caption_det": cap})
        else:
            merged.append({**fig, "_caption_det": None})

    for j, cap in enumerate(captions):  # keep orphan captions
        if j not in used:
            merged.append({**cap, "_caption_det": None})
    return merged

### Text extraction helpers

In [6]:
def extract_page_text_blocks(pdf_path: Path, page_index: int) -> list[dict]:
    """Extract normalised text blocks from one page via pdfminer."""
    blocks = []
    for page_layout in extract_pages(str(pdf_path), page_numbers=[page_index]):
        W, H = page_layout.width, page_layout.height
        for el in page_layout:
            if isinstance(el, LTTextBox):
                text = el.get_text().strip()
                if text:
                    x0, y0, x1, y1 = el.bbox
                    blocks.append({
                        "text":     text,
                        "box_norm": (x0 / W, 1 - y1 / H, x1 / W, 1 - y0 / H),
                    })
    return blocks


def overlap_fraction(text_box: tuple, det_box: tuple) -> float:
    """Fraction of text_box area that overlaps with det_box."""
    ax1, ay1, ax2, ay2 = text_box
    bx1, by1, bx2, by2 = det_box
    inter = max(0, min(ax2, bx2) - max(ax1, bx1)) * max(0, min(ay2, by2) - max(ay1, by1))
    area  = (ax2 - ax1) * (ay2 - ay1)
    return inter / area if area > 0 else 0.0


def get_caption_text(det: dict, text_blocks: list[dict], all_detections: list[dict]) -> str:
    """Retrieve caption text for a detected element."""
    det_box    = det["box_norm"]
    overlapping = [b for b in text_blocks if overlap_fraction(b["box_norm"], det_box) > 0.4]

    if det["class_id"] in CAPTION_CLASSES:
        return " ".join(b["text"] for b in overlapping)

    cx = (det_box[0] + det_box[2]) / 2
    for d2 in all_detections:
        if d2["class_id"] not in CAPTION_CLASSES:
            continue
        if abs(cx - (d2["box_norm"][0] + d2["box_norm"][2]) / 2) < 0.3:
            cap_blocks = [b for b in text_blocks if overlap_fraction(b["box_norm"], d2["box_norm"]) > 0.4]
            if cap_blocks:
                return cap_blocks[0]["text"]
    return ""

### Vision / LLM helpers

In [7]:
def _parse_vision_response(raw: str, label: str, fallback_counter: int) -> dict:
    """Parse structured LLM response into a metadata dict."""
    key_m     = re.search(r"KEY:\s*(.+)",          raw)
    heading_m = re.search(r"HEADING:\s*(.+)",      raw)
    content_m = re.search(r"CONTENT:\s*([\s\S]+)", raw)

    key     = key_m.group(1).strip()     if key_m     else f"{label.upper()}_{fallback_counter}"
    heading = heading_m.group(1).strip() if heading_m else ""
    content = content_m.group(1).strip() if content_m else raw
    content = re.sub(r"^HEADING:.*\n?", "", content).strip()

    kp = re.match(r"(FIGURE|TABLE)_(\d+)([a-z]?)", key, re.IGNORECASE)
    sub_plots = {
        m.group(1): m.group(2).strip()
        for m in re.finditer(r"\*\*\(([a-z])\)\*\*\s*([\s\S]*?)(?=\*\*\([a-z]\)\*\*|$)", content)
    }
    return {
        "key":       key,
        "heading":   heading,
        "type":      kp.group(1).upper() if kp else label.upper(),
        "number":    int(kp.group(2))    if kp else fallback_counter,
        "sub":       kp.group(3).lower() if kp and kp.group(3) else None,
        "content":   content,
        "sub_plots": sub_plots or None,
    }


async def vision_summary_async(
    img: Image.Image,
    label: str,
    caption_text: str,
    fallback_counter: int,
    loop: asyncio.AbstractEventLoop,
    executor: ThreadPoolExecutor,
) -> tuple[str, str, dict]:
    """Call the vision LLM asynchronously (Portkey is sync; offloaded to thread pool)."""
    b64 = pil_to_b64(img)

    def _call():
        return client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": [
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                    {"type": "text",      "text": USER_PROMPT_TEMPLATE.format(caption=caption_text)},
                ]},
            ],
            max_tokens=1200,
        )

    resp = await loop.run_in_executor(executor, _call)
    raw  = resp.choices[0].message.content.strip()
    meta = _parse_vision_response(raw, label, fallback_counter)
    meta["caption"] = caption_text
    return f"{meta['key']} : {meta['content']}", meta["key"], meta

### Page processor (async)

In [8]:
async def process_page(
    page_idx: int,
    total_pages: int,
    global_counter_start: int,
    use_vision: bool,
    dpi: int,
    loop: asyncio.AbstractEventLoop,
    executor: ThreadPoolExecutor,
) -> tuple[str, list[dict], int]:
    """
    Process one page: render → detect → extract text → call vision.
    All vision calls within a page run concurrently via asyncio.gather.
    Returns (page_markdown, figure_refs, n_detections_added).
    """
    print(f"\n--- Page {page_idx + 1}/{total_pages} ---")

    page_img    = await loop.run_in_executor(executor, page_to_pil, PDF_PATH, page_idx, dpi)
    raw_dets    = await loop.run_in_executor(executor, detect_visual_boxes, page_img)
    detections  = merge_figure_and_caption(raw_dets)
    text_blocks = await loop.run_in_executor(executor, extract_page_text_blocks, PDF_PATH, page_idx)

    print(f"  Detections: {len(detections)}")

    removed_ids   = set()
    vision_tasks  = []
    det_meta_list = []  # (det, first_block_idx, counter, label, caption_text)
    local_counter = global_counter_start

    for det in detections:
        overlapping = [
            (i, b) for i, b in enumerate(text_blocks)
            if overlap_fraction(b["box_norm"], det["box_norm"]) > 0.4
        ]
        if not overlapping:
            continue

        local_counter += 1
        caption_text    = get_caption_text(det, text_blocks, detections)
        label           = det["label"]
        first_block_idx = overlapping[0][0]

        for i, _ in overlapping:
            removed_ids.add(i)

        if use_vision:
            crop = crop_pil(page_img, det["box_norm"])
            vision_tasks.append(asyncio.ensure_future(
                vision_summary_async(crop, label, caption_text, local_counter, loop, executor)
            ))
            det_meta_list.append((first_block_idx, local_counter, label, caption_text))
        else:
            removed_ids.discard(first_block_idx)
            text_blocks[first_block_idx]["replacement"] = f"({label} {local_counter} — vision skipped)"

    # Fire all vision calls for this page in parallel
    vision_results = await asyncio.gather(*vision_tasks, return_exceptions=True)

    figure_refs = []
    for (first_block_idx, counter, label, caption_text), result in zip(det_meta_list, vision_results):
        removed_ids.discard(first_block_idx)
        if isinstance(result, Exception):
            replacement = f"({label} {counter} — vision unavailable: {result})"
            figure_refs.append({
                "key": f"{label.upper()}_{counter}", "heading": caption_text.split("\n")[0].strip(),
                "type": label.upper(), "number": counter, "sub": None,
                "caption": caption_text, "content": replacement, "sub_plots": None,
                "page": page_idx + 1,
            })
        else:
            replacement, json_key, meta = result
            figure_refs.append({**meta, "page": page_idx + 1})
            print(f"  {json_key} ✓")
        text_blocks[first_block_idx]["replacement"] = replacement

    lines = [
        b["replacement"] if "replacement" in b else b["text"]
        for i, b in enumerate(text_blocks)
        if "replacement" in b or i not in removed_ids
    ]
    page_md = f"## Page {page_idx + 1}\n\n" + "\n\n".join(lines)
    return page_md, figure_refs, local_counter - global_counter_start

### Main pipeline Execution

In [9]:
async def run_pipeline(use_vision: bool = True, dpi: int = 150, max_workers: int = 4):
    """
    Full async pipeline. Pages are processed sequentially (YOLO is CPU-heavy),
    but within each page all vision API calls run concurrently.
    """
    doc = fitz.open(str(PDF_PATH))
    total_pages = len(doc)
    doc.close()

    loop     = asyncio.get_event_loop()
    executor = ThreadPoolExecutor(max_workers=max_workers)

    all_figure_refs, markdown_pages, global_counter = [], [], 0

    for page_idx in range(total_pages):
        page_md, page_refs, added = await process_page(
            page_idx, total_pages, global_counter, use_vision, dpi, loop, executor
        )
        global_counter  += added
        markdown_pages.append(page_md)
        all_figure_refs.extend(page_refs)

    executor.shutdown(wait=False)
    return "\n\n---\n\n".join(markdown_pages), all_figure_refs


# Run — set use_vision=False to skip LLM calls
result_md, figure_refs = await run_pipeline(use_vision=True, dpi=150, max_workers=4)
print(f"\nDone! {len(figure_refs)} visuals processed.")


--- Page 1/5 ---

0: 1024x736 3 titles, 8 plain texts, 2 abandons, 1 figure, 1 figure_caption, 2710.5ms
Speed: 35.5ms preprocess, 2710.5ms inference, 5.6ms postprocess per image at shape (1, 3, 1024, 736)
  Detections: 1
  FIGURE_1 ✓

--- Page 2/5 ---

0: 1024x736 2 titles, 7 plain texts, 1 figure, 1 figure_caption, 1 isolate_formula, 1 formula_caption, 1968.1ms
Speed: 18.6ms preprocess, 1968.1ms inference, 4.8ms postprocess per image at shape (1, 3, 1024, 736)
  Detections: 1
  FIGURE_2 ✓

--- Page 3/5 ---

0: 1024x736 4 titles, 7 plain texts, 1 figure, 2 figure_captions, 1 isolate_formula, 1 formula_caption, 2694.2ms
Speed: 18.9ms preprocess, 2694.2ms inference, 0.0ms postprocess per image at shape (1, 3, 1024, 736)
  Detections: 2
  FIGURE_3 ✓
  FIGURE_3 ✓

--- Page 4/5 ---

0: 1024x736 3 titles, 5 plain texts, 1 figure, 1 figure_caption, 1 table, 1 table_caption, 2511.1ms
Speed: 0.0ms preprocess, 2511.1ms inference, 14.6ms postprocess per image at shape (1, 3, 1024, 736)
  Detecti

### Save outputs & preview

In [10]:
# Cell 11 — Save outputs & preview
md_path   = Path(OUTPUT_DIR) / f"{PDF_PATH.stem}.md"
json_path = Path(OUTPUT_DIR) / f"{PDF_PATH.stem}.json"

md_path.write_text(result_md, encoding="utf-8")
json_path.write_text(json.dumps(figure_refs, indent=2), encoding="utf-8")

print(f"Markdown → {md_path}")
print(f"JSON     → {json_path}")
print("\n" + "=" * 60)
print(result_md[:3000])

Markdown → Output\OP-RAG.md
JSON     → Output\OP-RAG.json

## Page 1

In Defense of RAG in the Era of Long-Context Language Models

Tan Yu
NVIDIA
Santa Clara, California
United States
tayu@nvidia.com

Anbang Xu
NVIDIA
Santa Clara, California
United States
anbangx@nvidia.com

Rama Akkiraju
NVIDIA
Santa Clara, California
United States
rakkiraju@nvidia.com

Abstract

Overcoming the limited context limitations in
early-generation LLMs, retrieval-augmented
generation (RAG) has been a reliable solution
for context-based answer generation in the past.
Recently, the emergence of long-context LLMs
allows the models to incorporate much longer
text sequences, making RAG less attractive.
Recent studies show that long-context LLMs
significantly outperform RAG in long-context
applications. Unlike the existing works favor-
ing the long-context LLM over RAG, we ar-
gue that the extremely long context in LLMs
suffers from a diminished focus on relevant in-
formation and leads to potential degradation i

In [25]:
import json
import re
from pathlib import Path

def enrich_markdown_with_figures(json_path, md_path, output_path=None):
    """
    Load figure JSON, build:
      Figure 4  -> content
      Figure 4a -> content
    Then enrich markdown references like 'Figure 4a' by adding the content after them.

    Existing markdown labels like:
      Figure 4:
      Figure 4 :
    are ignored.
    """

    # ---------- Load JSON ----------
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # ---------- Normalize figure items ----------
    # Case 1: top-level list
    if isinstance(data, list):
        figure_items = data

    # Case 2: top-level dict with "figures"
    elif isinstance(data, dict):
        if isinstance(data.get("figures"), list):
            figure_items = data["figures"]
        else:
            # fallback: maybe the dict itself is one figure-like object
            figure_items = [data]

    else:
        raise ValueError("Unsupported JSON structure. Expected a list or dict.")

    # ---------- Build figure dictionary ----------
    figure_dict = {}

    for fig in figure_items:
        if not isinstance(fig, dict):
            continue

        # Try multiple possible keys for figure number
        fig_num = (
            fig.get("figure_number")
            or fig.get("figure_num")
            or fig.get("number")
            or fig.get("fig_number")
            or ""
        )
        fig_num = str(fig_num).strip()

        # Try multiple possible keys for main content
        main_content = (
            fig.get("content")
            or fig.get("caption")
            or fig.get("description")
            or fig.get("text")
            or ""
        )
        main_content = str(main_content).strip()

        if not fig_num:
            continue

        # Add main figure mapping: Figure 4
        if main_content:
            figure_dict[f"Figure {fig_num}"] = main_content

        # Handle subplots if present
        sub_plots = fig.get("sub_plots") or fig.get("subplots") or []

        if isinstance(sub_plots, list):
            for sub in sub_plots:
                if not isinstance(sub, dict):
                    continue

                label = str(sub.get("label") or sub.get("sub") or "").strip()
                sub_content = (
                    sub.get("content")
                    or sub.get("caption")
                    or sub.get("description")
                    or sub.get("text")
                    or ""
                )
                sub_content = str(sub_content).strip()

                # normalize "a)" -> "a", "(a)" -> "a"
                label = re.sub(r"[^a-zA-Z0-9]", "", label)

                if label and sub_content:
                    figure_dict[f"Figure {fig_num}{label}"] = sub_content

        # Fallback if subplot label exists directly on main figure
        sub_label = str(fig.get("sub") or "").strip()
        sub_label = re.sub(r"[^a-zA-Z0-9]", "", sub_label)
        if sub_label and main_content:
            figure_dict[f"Figure {fig_num}{sub_label}"] = main_content

    # ---------- Load markdown ----------
    with open(md_path, "r", encoding="utf-8") as f:
        md_text = f.read()

    # ---------- Replacement ----------
    # Match:
    # Figure 4
    # Figure 4a
    # Figure 12b
    #
    # But do NOT match if immediately followed by optional spaces and colon
    # because we want to ignore:
    # Figure 4:
    # Figure 4 :
    pattern = r"\b(Figure\s+\d+[a-zA-Z]?)\b(?!\s*:)"
    
    def replace_match(match):
        ref = match.group(1)
        key = ref.strip()

        if key in figure_dict:
            return f"{ref} ({figure_dict[key]})"
        return ref

    enriched_text = re.sub(pattern, replace_match, md_text)

    # ---------- Save output ----------
    if output_path is None:
        md_path_obj = Path(md_path)
        output_path = str(md_path_obj.with_name(md_path_obj.stem + "_enriched.md"))

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(enriched_text)

    print(f"Enriched markdown saved to: {output_path}")
    print(f"Total figure mappings created: {len(figure_dict)}")

    # optional: return mapping for debugging
    return figure_dict


# ---------- USAGE ----------
json_file = r"Output\OP-RAG.json"
markdown_file = r"Output\OP-RAG.md"

figure_dict = enrich_markdown_with_figures(json_file, markdown_file)

# Optional: inspect first few mappings
for k, v in list(figure_dict.items())[:10]:
    print(k, "->", v[:120])

Enriched markdown saved to: Output\OP-RAG_enriched.md
Total figure mappings created: 4
Figure 1 -> ```markdown
| Method                     | EN.QA     |          | EN.MC     |          |
|----------------------------|-
Figure 2 -> The figure compares Vanilla RAG with the proposed order-preserve RAG using similarity scores of document chunks. A long 
Figure 3 -> The figure examines how context length affects the performance of RAG, with evaluations conducted on En.QA and EN.MC dat
Figure 4 -> Figure 4 shows comparisons between order-preserve RAG and vanilla RAG across two datasets.

**(a)** This sub-plot displa


In [30]:
import json

with open(r"Output\OP-RAG.json", "r", encoding="utf-8") as f:
    data = json.load(f)

figure_dict = {}

for fig in data:
    heading = fig.get("heading", "")
    content = fig.get("content", "")

    if heading and content:
        figure_dict[heading] = content

    sub_plots = fig.get("sub_plots") or {}

    for k, v in sub_plots.items():
        figure_dict[f"{heading}{k}"] = v

In [31]:
print(figure_dict)

{'Figure 1': 'Combined figure showing a comparison of order-preserve retrieval-augmented generation and long-context language models.\n\n**(a)** This plot presents the F1 scores of different models. The Llama3.1-70B OP-RAG variants (16K, 24K, 48K) outperform the long-context models (Llama3.1-70B, GPT-40, Gemini-1.5-Pro), with the highest score being 47.25 for OP-RAG-48K. The legend differentiates between Llama3.1-70B OP-RAG and Long Context approaches.\n\n**(b)** This plot shows the average input token count for the same models. OP-RAG models have significantly lower input token counts (e.g., 16K for OP-RAG-16K) compared to long-context models, which range from 117K to 196K tokens.\n```', 'Figure 1a': 'This plot presents the F1 scores of different models. The Llama3.1-70B OP-RAG variants (16K, 24K, 48K) outperform the long-context models (Llama3.1-70B, GPT-40, Gemini-1.5-Pro), with the highest score being 47.25 for OP-RAG-48K. The legend differentiates between Llama3.1-70B OP-RAG and L